In [2]:
import numpy as np
import pandas as pd
from itertools import product

# Quark masses (MeV, PDG 2022, MSbar, 2 GeV)
quark_masses = {
    'up': 2.16,
    'down': 4.67,
    'strange': 93,
    'charm': 1270,
    'bottom': 4180,
    'top': 172690
}

# All pairwise ratios (excluding self-ratios)
ratios = {}
for denom in quark_masses:
    for numer in quark_masses:
        if numer != denom:
            ratios[f"{numer}/{denom}"] = quark_masses[numer] / quark_masses[denom]

# Mathematical constants (small set)
consts = {
    'pi': np.pi,
    'e': np.e,
    'phi': (1 + np.sqrt(5)) / 2,
    'ln2': np.log(2)
}

A_vals = [1, 2, 6, 12, 24, 120]
powers = range(-3, 4)  # small exponents

results = []

for rname, rval in ratios.items():
    for (c1, v1), (c2, v2) in product(consts.items(), repeat=2):
        for a, b in product(powers, repeat=2):
            if a == b == 0:
                continue
            for A in A_vals:
                try:
                    val = A * (v1**a) * (v2**b)
                    relerr = abs(val - rval) / rval
                    complexity = abs(a) + abs(b) + (A != 1)
                    score = relerr + 0.01 * complexity  # Simplicity penalty
                    formula = f"{A}*{c1}^{a}*{c2}^{b}"
                    results.append((rname, score, relerr, complexity, formula, val))
                except Exception:
                    continue

df = pd.DataFrame(results, columns=['Ratio', 'Score', 'Rel. Error', 'Complexity', 'Formula', 'Value'])
df = df.sort_values(['Ratio', 'Score'])

# Deduplicate by value (to avoid algebraic duplicates)
df = df.round({'Value': 4})
df = df.drop_duplicates(subset=['Ratio', 'Value'])

# Show only the best (lowest score) for each ratio
print(df.groupby('Ratio').head(1)[['Ratio', 'Formula', 'Value', 'Rel. Error', 'Complexity']].to_string(index=False))

         Ratio         Formula      Value  Rel. Error  Complexity
  bottom/charm    2*pi^0*phi^1     3.2361    0.016793           2
   bottom/down    120*pi^0*e^2   886.6867    0.009372           3
bottom/strange    24*ln2^1*e^1    45.2201    0.006092           3
    bottom/top    1*e^-3*ln2^2     0.0239    0.011769           5
     bottom/up  120*phi^1*pi^2  1916.3226    0.009747           4
  charm/bottom   2*pi^-1*ln2^2     0.3059    0.006708           4
    charm/down    24*pi^3*e^-1   273.7577    0.006652           5
 charm/strange    2*pi^2*ln2^1    13.6822    0.001923           4
     charm/top  1*pi^-3*phi^-3     0.0076    0.035264           6
      charm/up     6*pi^2*pi^2   584.4545    0.005967           5
   down/bottom   1*pi^-3*pi^-3     0.0010    0.068978           6
    down/charm   1*pi^-3*pi^-2     0.0033    0.111336           5
  down/strange     1*pi^0*e^-3     0.0498    0.008523           3
      down/top   1*pi^-3*pi^-3     0.0010   37.463701           6
       dow